### Session 3: The ReAct Loop (Reason + Act) - The Heart of Agency 

Goal: Move from single tool call to multi-step reasoning.

In [1]:
# Install once per session
!pip install -q groq

from google.colab import userdata
from groq import Groq


# "gsk_WORKSHOP_KEY" is your actual API key (Secret)set in Colab.
api_key = userdata.get("gsk_WORKSHOP_KEY")

client = Groq(api_key=api_key)

def ask_llm(question):
    try:
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=[{"role": "user", "content": question}]
        )
        return response.choices[0].message.content
    except Exception as e:
        # This prints the FULL error details
        print(f"Error Type: {type(e).__name__}")
        print(f"Error Message: {str(e)}")
        
        # If it's an API error, get more details
        if hasattr(e, 'response'):
            print(f"Status Code: {e.response.status_code}")
            print(f"Response Body: {e.response.text}")
        return None

In [2]:
# Tool Definitions
import json

# 1. Define the Python Tool
def get_weather(city: str):
    # Mock database for workshop
    mock_db = {"mumbai": "32C, Humid", "delhi": "28C, Smoggy", "bangalore": "27C, Sultry"}
    return mock_db.get(city.lower(), "Data not available")

# 2. Write the TOOL SCHEMA (This is the tricky part students learn)
tool_schema = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get current weather of a city",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name",
                    }
                },
                "required": ["city"],
            },
        },
    }
]

# 3. The AGENTIC CALL
def agent_ask(user_query):
    response = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[{"role": "user", "content": user_query}],
        tools=tool_schema, # <--- THE MAGIC LINE
        tool_choice="auto"
    )
    return response


In [3]:

def run_agent(user_query):
    messages = [{"role": "user", "content": user_query}]
    
    # The Loop - Students fill in the condition
    while True:  # <--- STUDENTS FILL THIS
        response = client.chat.completions.create(
            model="openai/gpt-oss-20b",
            messages=messages,
            tools=tool_schema,
            tool_choice="auto"
        )
        
        response_message = response.choices[0].message
        messages.append(response_message) # Append assistant thought
        
        # Check if LLM wants to call a tool
        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                function_name = tool_call.function.name
                function_args = json.loads(tool_call.function.arguments)
                
                # EXECUTE THE PYTHON FUNCTION (Student fills this)
                if function_name == "get_weather":
                    result = get_weather(function_args.get("city"))
                    # Feed result back to LLM
                    messages.append({
                        "role": "tool",
                        "content": result,
                        "tool_call_id": tool_call.id
                    })
        else:
            # NO MORE TOOLS -> ANSWER IS READY
            return response_message.content



In [4]:
print(run_agent("Is it warmer in Mumbai or Delhi right now?"))

Based on the weather information, it seems that Mumbai is warmer than Delhi right now. The temperature in Mumbai is 32C while in Delhi it is 28C.
